In [30]:
# !pip install C:\Users\archi\Downloads\TA_Lib-0.4.28-cp310-cp310-win_amd64.whl

In [31]:
from IPython.display import display
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import logging
import talib

In [32]:
%run 01_config.ipynb

2024-10-29 17:09:09,497 - INFO - 正在讀取數據文件: D:\Min\Python\Project\FA_Data\meta_data\all_stocks_data.csv



正在載入台積電 (2330) 的數據...


2024-10-29 17:09:20,391 - INFO - 原始數據形狀: (2450900, 29)
2024-10-29 17:09:20,572 - INFO - 過濾後的數據形狀 (股票 2330): (2584, 29)
2024-10-29 17:09:20,583 - INFO - 開始日期過濾後的數據形狀: (437, 29)
2024-10-29 17:09:20,587 - INFO - 結束日期過濾後的數據形狀: (239, 29)



=== 股票數據摘要 ===
股票代碼: 2330
股票名稱: 台積電
數據期間: 2023-01-03 到 2023-12-29
交易天數: 239 天

--- 價格統計 ---
平均收盤價: 543.45
最高價: 594.00
最低價: 443.00
價格波動範圍: 151.00
標準差: 29.62

--- 交易統計 ---
平均日成交量: 27,535,566.40
平均日成交筆數: 29,635.65
平均日成交金額: 14,980,440,520.00

--- 技術指標概覽 ---

TREND類指標:
SMA30        - 均值: 536.47, 標準差: 28.85
DEMA30       - 均值: 543.00, 標準差: 29.66
EMA30        - 均值: 536.41, 標準差: 28.16
TSF          - 均值: 543.78, 標準差: 31.83
middleband   - 均值: 566.27, 標準差: 26.47

MOMENTUM類指標:
RSI          - 均值: 53.59, 標準差: 10.37
MACD         - 均值: 3.21, 標準差: 7.07
MACD_signal  - 均值: 3.12, 標準差: 6.46
MACD_hist    - 均值: 0.09, 標準差: 2.59

VOLATILITY類指標:
slowk        - 均值: 52.71, 標準差: 28.52
slowd        - 均值: 52.39, 標準差: 25.99
SAR          - 均值: 539.43, 標準差: 34.31

=== 數據載入完成 ===


In [33]:
class FeatureGenerator:
    """特徵生成器"""
    def __init__(self):
        self.scaler = StandardScaler()
        # 使用DataConfig中的參數
        self.tech_params = DataConfig.INDICATOR_PARAMS
        self.tech_features = DataConfig.TECHNICAL_INDICATORS
        
    def generate_basic_features(self, df):
        """生成基本技術指標"""
        try:
            df_features = df.copy()
            
            # 轉換數值型別
            numeric_columns = ['收盤價', '開盤價', '最高價', '最低價', '成交股數', '本益比']
            for col in numeric_columns:
                df_features[col] = pd.to_numeric(df_features[col], errors='coerce')
            
            # 計算移動平均 (使用配置參數)
            period = self.tech_params['SMA']['timeperiod']
            df_features['SMA30'] = df_features['收盤價'].rolling(window=period).mean()
            df_features['DEMA30'] = talib.DEMA(df_features['收盤價'], timeperiod=period)
            df_features['EMA30'] = talib.EMA(df_features['收盤價'], timeperiod=period)
            
            # RSI
            rsi_period = self.tech_params['RSI']['timeperiod']
            df_features['RSI'] = talib.RSI(df_features['收盤價'], timeperiod=rsi_period)
            
            # MACD
            macd_params = self.tech_params['MACD']
            df_features['MACD'], df_features['MACD_signal'], df_features['MACD_hist'] = talib.MACD(
                df_features['收盤價'],
                fastperiod=macd_params['fastperiod'],
                slowperiod=macd_params['slowperiod'],
                signalperiod=macd_params['signalperiod']
            )
            
            # KD
            kd_params = self.tech_params['KD']
            df_features['slowk'], df_features['slowd'] = talib.STOCH(
                df_features['最高價'],
                df_features['最低價'],
                df_features['收盤價'],
                fastk_period=kd_params['fastk_period'],
                slowk_period=kd_params['slowk_period'],
                slowd_period=kd_params['slowd_period']
            )
            
            return df_features
            
        except Exception as e:
            logging.error(f"生成基本特徵時出錯: {str(e)}")
            raise

    def generate_advanced_features(self, df):
        """生成進階特徵"""
        try:
            df_features = df.copy()
            
            # 趨勢強度指標
            df_features['趨勢強度'] = np.where(
                df_features['收盤價'] > df_features['SMA30'],
                (df_features['收盤價'] - df_features['SMA30']) / df_features['SMA30'] * 100,
                -(df_features['SMA30'] - df_features['收盤價']) / df_features['SMA30'] * 100
            )
            
            # 波動率指標
            df_features['波動率'] = df_features['收盤價'].rolling(window=20).std() / df_features['收盤價'].rolling(window=20).mean()
            
            # 其他特徵
            df_features['量能趨勢'] = df_features['成交股數'].rolling(window=5).mean() / df_features['成交股數'].rolling(window=20).mean()
            df_features['均線糾結度'] = pd.DataFrame({
                'SMA30': df_features['SMA30'],
                'DEMA30': df_features['DEMA30'],
                'EMA30': df_features['EMA30']
            }).std(axis=1) / df_features['收盤價']
            
            df_features['振幅'] = (df_features['最高價'] - df_features['最低價']) / df_features['開盤價'] * 100
            df_features['量比'] = df_features['成交股數'] / df_features['成交股數'].rolling(window=5).mean()
            df_features['RSI_動能'] = df_features['RSI'].diff() / df_features['RSI'].shift(1)
            df_features['MACD_動能'] = df_features['MACD'].diff() / df_features['MACD'].shift(1)
            df_features['KD_差值'] = df_features['slowk'] - df_features['slowd']
            df_features['本益比_相對值'] = df_features['本益比'] / df_features['本益比'].rolling(window=30).mean()
            
            # 技術綜合評分
            df_features['技術綜合評分'] = (
                (df_features['RSI'] / 100 * 0.15) +
                ((df_features['MACD'] - df_features['MACD'].min()) / 
                 (df_features['MACD'].max() - df_features['MACD'].min()) * 0.15) +
                (df_features['slowk'] / 100 * 0.15) +
                (df_features['量能趨勢'] * 0.15) +
                ((df_features['趨勢強度'] - df_features['趨勢強度'].min()) / 
                 (df_features['趨勢強度'].max() - df_features['趨勢強度'].min()) * 0.15) +
                ((df_features['本益比_相對值'] - df_features['本益比_相對值'].min()) /
                 (df_features['本益比_相對值'].max() - df_features['本益比_相對值'].min()) * 0.25)
            )
            
            return df_features
            
        except Exception as e:
            logging.error(f"生成進階特徵時出錯: {str(e)}")
            raise
    
    def prepare_training_data(self, df):
        """準備訓練數據"""
        try:
            # 生成所有特徵
            df_basic = self.generate_basic_features(df)
            df_features = self.generate_advanced_features(df_basic)
            
            # 選擇最終特徵
            feature_columns = [
                '趨勢強度', '均線糾結度', '波動率', '振幅',
                '量能趨勢', '量比', 'RSI_動能', 'MACD_動能',
                'RSI', 'MACD', 'KD_差值', '本益比_相對值',
                '技術綜合評分'
            ]
            
            features = df_features[feature_columns].copy()
            
            # 處理缺失值
            features = features.fillna(method='ffill').fillna(method='bfill')
            
            # 標準化特徵
            features_scaled = self.scaler.fit_transform(features)
            
            return features_scaled, feature_columns
        
        except Exception as e:
            logging.error(f"準備訓練數據時出錯: {str(e)}")
            raise

In [34]:
# 測試代碼
if __name__ == "__main__":
    try:
        # 使用load_stock_data載入測試數據
        test_stock_id = '2330'
        test_start_date = '2023-01-01'
        test_end_date = '2024-01-01'
        
        print(f"\n正在處理 {test_stock_id} 的特徵...")
        
        # 載入數據
        df = load_stock_data(test_stock_id, test_start_date, test_end_date)
        
        # 初始化特徵生成器
        generator = FeatureGenerator()
        
        # 生成特徵
        features, feature_names = generator.prepare_training_data(df)
        
        print("\n=== 特徵生成完成 ===")
        print(f"特徵數量: {len(feature_names)}")
        print("特徵名稱:", feature_names)
        
        # 顯示特徵統計
        features_df = pd.DataFrame(features, columns=feature_names)
        print("\n特徵統計:")
        print(features_df.describe())
        
    except Exception as e:
        logging.error(f"測試過程發生錯誤: {str(e)}")

2024-10-29 17:09:21,041 - INFO - 正在讀取數據文件: D:\Min\Python\Project\FA_Data\meta_data\all_stocks_data.csv



正在處理 2330 的特徵...


2024-10-29 17:09:30,634 - INFO - 原始數據形狀: (2450900, 29)
2024-10-29 17:09:30,830 - INFO - 過濾後的數據形狀 (股票 2330): (2584, 29)
2024-10-29 17:09:30,833 - INFO - 開始日期過濾後的數據形狀: (437, 29)
2024-10-29 17:09:30,836 - INFO - 結束日期過濾後的數據形狀: (239, 29)



=== 股票數據摘要 ===
股票代碼: 2330
股票名稱: 台積電
數據期間: 2023-01-03 到 2023-12-29
交易天數: 239 天

--- 價格統計 ---
平均收盤價: 543.45
最高價: 594.00
最低價: 443.00
價格波動範圍: 151.00
標準差: 29.62

--- 交易統計 ---
平均日成交量: 27,535,566.40
平均日成交筆數: 29,635.65
平均日成交金額: 14,980,440,520.00

--- 技術指標概覽 ---

TREND類指標:
SMA30        - 均值: 536.47, 標準差: 28.85
DEMA30       - 均值: 543.00, 標準差: 29.66
EMA30        - 均值: 536.41, 標準差: 28.16
TSF          - 均值: 543.78, 標準差: 31.83
middleband   - 均值: 566.27, 標準差: 26.47

MOMENTUM類指標:
RSI          - 均值: 53.59, 標準差: 10.37
MACD         - 均值: 3.21, 標準差: 7.07
MACD_signal  - 均值: 3.12, 標準差: 6.46
MACD_hist    - 均值: 0.09, 標準差: 2.59

VOLATILITY類指標:
slowk        - 均值: 52.71, 標準差: 28.52
slowd        - 均值: 52.39, 標準差: 25.99
SAR          - 均值: 539.43, 標準差: 34.31

=== 特徵生成完成 ===
特徵數量: 13
特徵名稱: ['趨勢強度', '均線糾結度', '波動率', '振幅', '量能趨勢', '量比', 'RSI_動能', 'MACD_動能', 'RSI', 'MACD', 'KD_差值', '本益比_相對值', '技術綜合評分']

特徵統計:
               趨勢強度         均線糾結度           波動率            振幅          量能趨勢  \
count  2.390000e+02  2.390000e+0